# Semi-Parametric Regression: Concrete Compressive Strength Benchmark (Table 4)
This notebook evaluates the hybrid semi-parametric models on the **Concrete Compressive Strength** dataset (UCI ID: 165) using 5-fold cross-validation.


In [1]:
import sys
sys.path.append('..')

import numpy as np
from sklearn.model_selection import KFold
from src.data_loader import load_concrete_dataset, prepare_pca_features
from src.hybrid_model import HybridBPWNN
from src.metrics import print_metrics, aggregate_cv_results


### 1. Load Data & Preprocessing
We scale features and target independently, then apply PCA to extract 4 orthogonal principal components.


In [2]:
X_raw, y_raw, names = load_concrete_dataset()
print(f'Instances: {len(X_raw)}, Features: {X_raw.shape[1]}')

X_pca, y_scaled, scaler_x, scaler_y, pca = prepare_pca_features(X_raw, y_raw, n_components=4)
X_param = X_pca[:, [0, 1, 2]]
X_nonparam = X_pca[:, 3]


Instances: 1030, Features: 8


### 2. Run 5-Fold Cross Validation


In [3]:
models_to_run = [
    ('Ridge-NN', 'Ridge'),
    ('Lasso-NN', 'Lasso'),
    ('Elastic-NN', 'ElasticNet'),
    ('RBPW-NN', 'RBPW'),
    ('LBPW-NN', 'LBPW'),
    ('EBPW-NN', 'EBPW'),
]

model_metrics = {name: [] for name, _ in models_to_run}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_pca), 1):
    print(f'Fold {fold_idx} / 5...')
    X_tr_p, X_va_p = X_param[train_idx], X_param[val_idx]
    X_tr_np, X_va_np = X_nonparam[train_idx], X_nonparam[val_idx]
    y_tr, y_va = y_scaled[train_idx], y_scaled[val_idx]

    for display_name, m_type in models_to_run:
        hybrid = HybridBPWNN(model_type=m_type, hidden_layers=1, hidden_nodes=100, epochs=50, verbose=0)
        y_pred_tr, y_pred_va, num_params = hybrid.fit_and_predict(X_tr_p, X_va_p, X_tr_np, X_va_np, y_tr)
        metrics = print_metrics(y_tr, y_va, y_pred_tr, y_pred_va, display_name, num_params, scaler_y=scaler_y)
        model_metrics[display_name].append(metrics)

summary = aggregate_cv_results(model_metrics)


Fold 1 / 5...
[Ridge-NN            ] Test MSE: 189.8803 | R²: 0.2631 | AIC: 1088.76 | BIC: 1102.07
[Lasso-NN            ] Test MSE: 189.2107 | R²: 0.2657 | AIC: 1088.03 | BIC: 1101.34
[Elastic-NN          ] Test MSE: 189.2107 | R²: 0.2657 | AIC: 1088.03 | BIC: 1101.34


ValueError: Unknown model_type: RBPW